In [ ]:
import sys
import os

# On remonte d'un cran pour atteindre la racine du projet
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)


from src.O1_preprocessing.split_data import create_split
from src.O1_preprocessing.loader import get_available_patients

In [ ]:
from src.utils.io import get_data_path
inventory = get_available_patients(str(get_data_path()))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 01 : Imports et Configuration
# ═══════════════════════════════════════════════════════════════
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

from src.O1_preprocessing.loader import get_available_patients, load_patient_data
from src.O2_classical_detection.detector import ClassicalDetector

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Environnement prêt pour la détection classique")



In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 : Imports et Configuration des Chemins
# ═══════════════════════════════════════════════════════════════
import sys
import os
from pathlib import Path

# ✅ Ajouter src/ au PYTHONPATH
src_path = Path(__file__).parent.parent / "src" if '__file__' in dir() else Path.cwd().parent / "src"
sys.path.insert(0, str(src_path))

# ✅ Utiliser ton utilitaire io.py
from src.utils.io import get_data_path, get_project_root

print(f"📂 Racine du projet : {get_project_root()}")
print(f"📁 Dossier data : {get_data_path()}")
#print(f"📁 Dossier src : {get_src_path()}")

# ✅ Vérifier que les dossiers existent
assert (get_data_path() / "raw").exists(), "❌ data/raw/ introuvable"
#assert (get_src_path() / "O1_preprocessing").exists(), "❌ src/O1_preprocessing/ introuvable"

print("✅ Tous les chemins sont valides !")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 : Initialisation du Détecteur
# ═══════════════════════════════════════════════════════════════
config = {
    'lung_hu_threshold': -400,
    'threshold_method': 'percentile',
    'threshold_percentile': 95,
    'min_hu': 50,
    'max_hu': 400,
    'min_candidate_volume': 20,
    'max_candidate_volume': 5000
}

detector = ClassicalDetector(config)
print(f"🔧 Détecteur configuré : {config}")


In [ ]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient 
# ══════════════════════════════════
from pathlib import Path
import sys
import os


NOTEBOOK_DIR = Path.cwd()  # Dossier actuel du notebook
PROJECT_ROOT = NOTEBOOK_DIR.parent  # Remonte d'un niveau depuis notebooks/
DATA_DIR = PROJECT_ROOT / "data"

# Vérifier que le dossier existe
if not DATA_DIR.exists():
    print(f"❌ Dossier data introuvable : {DATA_DIR}")
    print(f"📂 Dossier actuel : {NOTEBOOK_DIR}")
    print(f"💡 Essaye avec : DATA_DIR = Path('data') si ton notebook est à la racine")
else:
    print(f"✅ Dossier data trouvé : {DATA_DIR}")

# Lister le contenu pour debug
print(f"\n📁 Contenu de data/ :")
for item in DATA_DIR.iterdir():
    print(f"   - {item.name}")

# Charger l'inventaire avec le chemin absolu
inventory = get_available_patients(str(DATA_DIR))
print(f"\n📊 {len(inventory['with_both'])} patients disponibles")

if inventory['with_both']:
    test_id = inventory['with_both'][0]
    print(f"🧪 Test avec patient : {test_id}")
    
    # Charger les données (mode processed si disponible)
    vol, mask, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        print(f"📦 Volume chargé : {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]")
        
        # Lancer la détection
        result = detector.detect(vol)
        
        print(f"\n🎯 Résultats :")
        print(f"   • Voxels pulmonaires : {result['lung_mask'].sum()}")
        print(f"   • Candidats détectés : {result['n_candidates']}")
        print(f"   • Voxels candidats : {result['total_candidate_voxels']}")
        
        # Afficher les features des candidats
        if len(result['candidates_df']) > 0:
            print(f"\n📋 Top 5 candidats par volume :")
            display(result['candidates_df'].nlargest(5, 'volume_voxels')[
                ['region_id', 'volume_voxels', 'intensity_mean', 'sphericity']
            ])
    else:
        print(f"❌ Échec de chargement du patient {test_id}")
else:
    print(f"\n❌ Aucun patient trouvé !")
    print(f"💡 Vérifie que :")
    print(f"   1. Le dossier data/raw/CT_scans_* existe")
    print(f"   2. Le dossier data/raw/GroundTruth/ existe")
    print(f"   3. Les dossiers PAT* contiennent des fichiers .dcm")

In [ ]:
# Cellule 3 : Test sur un patient réel
inventory = get_available_patients("../data")
if inventory['with_both']:
    test_id = inventory['with_both'][0]
    vol, mask, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        result = detector.detect(vol, spacing=(1.0, 0.7, 0.7))
        print(f"🎯 {result['n_candidates']} candidats détectés")
        display(result['candidates_df'].head())

In [ ]:
"""
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
from src.O1_preprocessing.loader import get_available_patients, load_patient_data
from src.O2_classical_detection.lung_segmentation import segment_lungs, visualize_lung_segmentation

# Charger un patient réel
inventory = get_available_patients('data')
if inventory['with_both']:
    vol, mask_gt, meta = load_patient_data(inventory['with_both'][0], inventory)
    
    print(f'📦 Volume: {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]')
    
    # Segmenter
    lung_mask = segment_lungs(vol, method='threshold', hu_threshold=-400)
    
    print(f'✅ Poumons: {lung_mask.sum()} voxels ({100*lung_mask.sum()/vol.size:.1f}%)')
    
    # Visualiser (optionnel, ouvre une fenêtre matplotlib)
    # visualize_lung_segmentation(vol, lung_mask, num_slices=5)

"""

In [ ]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient (VERSION FINALE)
# ═════════════════════════════════
from pathlib import Path
import sys
import os

# ✅ Calculer le chemin vers le dossier 'data'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

# Vérifier que le dossier existe
if not DATA_DIR.exists():
    print(f"❌ Dossier data introuvable : {DATA_DIR}")
else:
    print(f"✅ Dossier data trouvé : {DATA_DIR}")

# Lister le contenu pour debug
print(f"\n📁 Contenu de data/ :")
for item in DATA_DIR.iterdir():
    print(f"   - {item.name}")

# ✅ Charger l'inventaire (laisse le loader choisir raw/processed)
inventory = get_available_patients(str(DATA_DIR))

print(f"\n📊 {len(inventory['with_both'])} patients disponibles")
print(f"📂 Mode utilisé : {'PROCESSED' if inventory.get('use_processed', False) else 'RAW'}")

if inventory['with_both']:
    test_id = inventory['with_both'][0]
    print(f"🧪 Test avec patient : {test_id}")
    
    # Charger les données
    vol, mask_gt, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        print(f"📦 Volume chargé : {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]")
        
        # ✅ Configuration avec seuils adaptés au mode
        if inventory.get('use_processed', False):
            # Mode PROCESSED : valeurs normalisées [0, 1]
            print("⚠️ Mode PROCESSED détecté - adaptation des seuils...")
            config = {
                'lung_hu_threshold': 0.17,   # ≈ -400 HU normalisé
                'threshold_method': 'percentile',
                'threshold_percentile': 95,
                'min_hu': 0.54,               # ≈ 50 HU normalisé
                'max_hu': 0.83,               # ≈ 400 HU normalisé
                'min_candidate_volume': 20,
                'max_candidate_volume': 5000
            }
            # Spacing par défaut (1mm isotrope si preprocessé correctement)
            spacing = (1.0, 1.0, 1.0)
        else:
            # Mode RAW : valeurs HU réelles
            config = {
                'lung_hu_threshold': -400,    # HU réels
                'threshold_method': 'percentile',
                'threshold_percentile': 95,
                'min_hu': 50,                  # HU réels
                'max_hu': 400,                 # HU réels
                'min_candidate_volume': 20,
                'max_candidate_volume': 5000
            }
            # Spacing depuis les métadonnées DICOM
            spacing = (
                meta.get('slice_spacing', 1.0),
                meta.get('spacing', [1.0, 1.0])[0],
                meta.get('spacing', [1.0, 1.0])[1]
            )
        
        detector = ClassicalDetector(config)
        
        # Lancer la détection
        result = detector.detect(vol, spacing=spacing)
        
        print(f"\n🎯 Résultats :")
        print(f"   • Voxels pulmonaires : {result['lung_mask'].sum()}")
        print(f"   • Candidats détectés : {result['n_candidates']}")
        print(f"   • Voxels candidats : {result['total_candidate_voxels']}")
        
        if len(result['candidates_df']) > 0:
            print(f"\n📋 Top 5 candidats par volume :")
            display(result['candidates_df'].nlargest(5, 'volume_voxels')[
                ['region_id', 'volume_voxels', 'intensity_mean', 'sphericity']
            ])
        else:
            print(f"\n💡 Aucun candidat détecté → Ajuster les seuils dans config")
    else:
        print(f"❌ Échec de chargement du patient {test_id}")
else:
    print(f"\n❌ Aucun patient trouvé !")

### Forcer le Mode RAW Sans grande Modification

In [ ]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient 
# ═════════════════════════════════
from pathlib import Path
import sys
import os

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

# ✅ Cette ligne va maintenant charger en RAW car processed/ n'existe plus
inventory = get_available_patients(str(DATA_DIR))

print(f"\n📊 {len(inventory['with_both'])} patients disponibles")
print(f"📂 Mode utilisé : {'PROCESSED' if inventory.get('use_processed', False) else 'RAW'}")
# → Doit afficher : Mode utilisé : RAW ✅

In [ ]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient 
# ═════════════════════════════════
from pathlib import Path
import sys
import os

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

if not DATA_DIR.exists():
    print(f"❌ Dossier data introuvable : {DATA_DIR}")
else:
    print(f"✅ Dossier data trouvé : {DATA_DIR}")

print(f"\n📁 Contenu de data/ :")
for item in DATA_DIR.iterdir():
    print(f"   - {item.name}")

# ✅ Charger l'inventaire (mode RAW car processed_backup)
inventory = get_available_patients(str(DATA_DIR))

print(f"\n📊 {len(inventory['with_both'])} patients disponibles")
print(f"📂 Mode utilisé : {'PROCESSED' if inventory.get('use_processed', False) else 'RAW'}")

if inventory['with_both']:
    test_id = inventory['with_both'][0]
    print(f"🧪 Test avec patient : {test_id}")
    
    # ✅ Charger les données RAW
    vol, mask_gt, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        print(f"📦 Volume chargé : {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]")
        
        # ✅ Configuration avec seuils HU RÉELS
        config = {
            'lung_hu_threshold': -400,
            'threshold_method': 'percentile',
            'threshold_percentile': 85,
            'min_hu': 20,
            'max_hu': 400,
            'min_candidate_volume': 20,
            'max_candidate_volume': 5000
        }
        detector = ClassicalDetector(config)
        
        # ✅ Spacing depuis métadonnées
        spacing = (
            meta.get('slice_spacing', 1.0),
            meta.get('spacing', [1.0, 1.0])[0],
            meta.get('spacing', [1.0, 1.0])[1]
        )
        result = detector.detect(vol, spacing=spacing)
        
        print(f"\n🎯 Résultats :")
        print(f"   • Voxels pulmonaires : {result['lung_mask'].sum()}")
        print(f"   • Candidats détectés : {result['n_candidates']}")
        print(f"   • Voxels candidats : {result['total_candidate_voxels']}")
        
        if len(result['candidates_df']) > 0:
            print(f"\n📋 Top 5 candidats par volume :")
            display(result['candidates_df'].nlargest(5, 'volume_voxels')[
                ['region_id', 'volume_voxels', 'intensity_mean', 'sphericity']
            ])
        else:
            print(f"\n💡 Aucun candidat détecté → Ajuster les seuils")
    else:
        print(f"❌ Échec de chargement du patient {test_id}")
else:
    print(f"\n❌ Aucun patient trouvé !")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE BATCH : Détection sur tous les 35 patients (RAW → candidates.npy)
# ═══════════════════════════════════════════════════════════════
from pathlib import Path
from src.O1_preprocessing.loader import get_available_patients
from src.O2_classical_detection.detector import ClassicalDetector

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# ✅ Inventaire RAW (force_raw=True garantit les HU vraies)
raw_inventory = get_available_patients(str(DATA_DIR), force_raw=True)
print(f"📊 {len(raw_inventory['with_both'])} patients RAW disponibles")

# ✅ Seuils HU physiologiques (données RAW [-1024, 3071])
config = {
    'lung_hu_threshold': -400,
    'threshold_method': 'percentile',
    'threshold_percentile': 85,
    'min_hu': 20,
    'max_hu': 400,
    'min_candidate_volume': 20,
    'max_candidate_volume': 5000
}
detector = ClassicalDetector(config)

# ✅ Boucle sur tous les patients
success, skip, fail = 0, 0, 0
for patient_id in raw_inventory['with_both']:
    result = detector.detect_and_save(
        patient_id=patient_id,
        processed_dir=str(PROCESSED_DIR),
        raw_inventory=raw_inventory,
        force=False  # Mettre True pour écraser les fichiers existants
    )
    if result is None:
        skip += 1
    else:
        success += 1

print(f"✅ Terminé : {success} traités, {skip} skippés")
print(f"📂 candidates.npy sauvegardés dans : {PROCESSED_DIR}")


#### Remaque:

Cela signifie que les "aspérités" ont ete bien supprimé aux bords des vaisseaux pour ne garder que le cœur des structures denses.